# 🗂️ Python Topological Sort — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Topological sort is like getting dressed in the morning — some steps must happen before others.
> You can't put on shoes before socks, or a belt before pants.
> Each task (node) has prerequisites (edges). Topological sort finds a valid order
> where every prerequisite comes before the task that needs it.
> If there's a cycle (A requires B requires A), no valid order exists.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Topological Sort? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — Both Algorithms](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Course Schedule — Can Finish (LC 207)](#5) |
| 6 | [Pattern 2: Course Schedule II — Find Order (LC 210)](#6) |
| 7 | [Pattern 3: Alien Dictionary (LC 269)](#7) |
| 8 | [The Topological Sort Decision Map](#8) |
| 9 | [Interview Cheat Sheet](#9) |

<a id='1'></a>
## 1. What Is Topological Sort? The Visual Model

```
               TOPOLOGICAL SORT — THE GETTING DRESSED PROBLEM

  DIRECTED ACYCLIC GRAPH (DAG):

   underwear ──► pants ──► belt ──► jacket
       │           │
       ▼           ▼
     socks       shoes ──► (done)

  Valid topological orders (many may exist):
    underwear → socks → pants → shoes → belt → jacket
    underwear → pants → socks → belt → shoes → jacket

  INVALID ORDER:  shoes → pants  (shoes require pants first!)

  ──────────────────────────────────────────────────────

  TWO ALGORITHMS:

  KAHN'S (BFS-based):                DFS POST-ORDER:
  1. Compute in-degree for each node  1. DFS each node, marking state
  2. Queue all nodes with in-degree=0 2. After all neighbors visited,
  3. Process queue:                      add node to result list
     pop node → add to result          3. Reverse the result
     decrement neighbor in-degrees   Cycle = a node is visited while
     re-queue neighbors at 0            still in-progress
  Cycle = queue empties early

  IN-DEGREE = number of incoming edges (dependencies that must finish first)
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
from collections import defaultdict, deque

# Build a directed graph from edge list: (prerequisite, course)
# Edge a→b means: a must come before b
def build_graph(num_nodes, edges):
    adj = defaultdict(list)      # adjacency list: node → [nodes it points to]
    indegree = [0] * num_nodes   # indegree[i] = how many nodes point INTO node i
    for prereq, course in edges:
        adj[prereq].append(course)  # edge: prereq → course
        indegree[course] += 1       # course has one more incoming dependency
    return adj, indegree

# Example: 4 courses, prerequisites: 0→1, 0→2, 1→3, 2→3
# (must take 0 before 1, 0 before 2, 1 and 2 before 3)
adj, indegree = build_graph(4, [(0,1),(0,2),(1,3),(2,3)])
print("adjacency list:", dict(adj))
print("in-degrees:     ", indegree)  # [0, 1, 1, 2]
print()

# Kahn's BFS-based topological sort
def kahns_topo_sort(num_nodes, edges):
    adj, indegree = build_graph(num_nodes, edges)
    queue = deque([i for i in range(num_nodes) if indegree[i] == 0])  # sources first
    order = []
    while queue:
        node = queue.popleft()    # node with no more unresolved dependencies
        order.append(node)
        for neighbor in adj[node]:
            indegree[neighbor] -= 1         # dependency resolved
            if indegree[neighbor] == 0:
                queue.append(neighbor)      # ready to process
    # if order length < num_nodes, a cycle exists
    return order if len(order) == num_nodes else []

print("Kahn's sort:", kahns_topo_sort(4, [(0,1),(0,2),(1,3),(2,3)]))
print("Kahn's with cycle:", kahns_topo_sort(2, [(0,1),(1,0)]))  # [] = cycle
print("graph setup complete.")

<a id='3'></a>
## 3. The Core API — Both Algorithms

```
ALGORITHM     APPROACH          CYCLE DETECTION      WHEN TO USE
─────────────────────────────────────────────────────────────────────
Kahn's BFS    in-degree queue   order length < n     prefer for clean code
DFS post-order  recursive DFS   in-progress set      needed when DFS required

KAHN'S ALGORITHM — O(V+E):
  1. Build adj list + compute in-degrees
  2. Queue all zero-in-degree nodes (no dependencies)
  3. Pop → add to order → decrement neighbor in-degrees → re-queue if 0
  4. Cycle = len(order) < num_nodes (some nodes never reached 0)

DFS POST-ORDER — O(V+E):
  States: WHITE=unvisited, GRAY=in-progress, BLACK=done
  1. DFS each unvisited node
  2. On entering: mark GRAY (in progress)
  3. If a GRAY neighbor is encountered: CYCLE detected
  4. After all neighbors done: mark BLACK, append to result
  5. Reverse result = topological order

THINGS YOU DO NOT DO:
❌  Apply topological sort to an undirected graph
❌  Assume there is only one valid topological order
❌  Forget to check for cycles — always validate len(order)==n
❌  Confuse in-degree (incoming edges) with out-degree (outgoing edges)
```

In [ ]:
# DFS POST-ORDER topological sort with cycle detection
def dfs_topo_sort(num_nodes, edges):
    adj, _ = build_graph(num_nodes, edges)

    WHITE, GRAY, BLACK = 0, 1, 2      # unvisited, in-progress, completed
    color = [WHITE] * num_nodes
    order = []
    has_cycle = [False]

    def dfs(node):
        if has_cycle[0]:
            return
        color[node] = GRAY             # mark as in-progress (we're inside this node)
        for neighbor in adj[node]:
            if color[neighbor] == GRAY:
                has_cycle[0] = True    # back edge = cycle in directed graph
                return
            if color[neighbor] == WHITE:
                dfs(neighbor)          # explore unvisited neighbor
        color[node] = BLACK            # all descendants done — safe to finalize
        order.append(node)             # post-order: add AFTER all children done

    for i in range(num_nodes):
        if color[i] == WHITE:
            dfs(i)                     # handle disconnected components

    if has_cycle[0]:
        return []                      # no valid order exists
    return order[::-1]                 # reverse post-order = topological order

# Demo
print("DFS post-order:", dfs_topo_sort(4, [(0,1),(0,2),(1,3),(2,3)]))
print("DFS with cycle: ", dfs_topo_sort(2, [(0,1),(1,0)]))  # []
print("Both algorithms demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
──────────────────────────────────────────────────────────────────────
"can all courses be finished"           Kahn's — check len(order)==n
"find a valid order for courses"        Kahn's — return the order list
"detect cycle in directed graph"        Either — cycle ↔ len(order)<n or GRAY neighbor
"alien dictionary / char ordering"     Topo sort on character dependency graph
"task scheduling with dependencies"    Kahn's BFS (natural layer-by-layer)
"build order / compilation order"      Topo sort with reverse edges
"shortest path in DAG"                 Topo sort then relax edges in order
"count paths from source to dest DAG"  Topo sort then DP forward
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Course Schedule — Can Finish — LC 207

---

```
PROBLEM:
  Given numCourses and a list of [course, prerequisite] pairs,
  determine if it's possible to finish all courses.
  (Equivalent to: does the prerequisite graph have a cycle?)

TRICK:
  Run Kahn's algorithm. If the topological order contains all n nodes,
  no cycle exists and all courses can be finished.
  If the order is shorter, some courses form a cycle — impossible.

SLOW MOTION TRACE on numCourses=4, prerequisites=[[1,0],[2,0],[3,1],[3,2]]:

  Edges: 0→1, 0→2, 1→3, 2→3
  in-degrees: [0, 1, 1, 2]

  queue = [0]   (only 0 has in-degree 0)

  step  pop  order    indegree changes          queue after
    1    0   [0]     in[1]-=1=0, in[2]-=1=0    [1, 2]
    2    1   [0,1]   in[3]-=1=1                [2]
    3    2   [0,1,2] in[3]-=1=0                [3]
    4    3   [0,1,2,3]  (no neighbors)         []

  len(order)=4 == numCourses=4 → True (no cycle)

SLOW MOTION TRACE on numCourses=2, prerequisites=[[0,1],[1,0]] (cycle):

  Edges: 1→0, 0→1
  in-degrees: [1, 1]
  queue = []  (nothing starts at 0) → order stays empty → False

KEY INSIGHT:
  A cycle means some nodes NEVER reach in-degree 0 — Kahn's queue never processes them.

TIME:  O(V+E) — build graph + BFS each node/edge once
SPACE: O(V+E) — adjacency list + in-degree array + queue
```

In [ ]:
def can_finish(num_courses, prerequisites):
    """
    LC 207 — Course Schedule
    Approach: Kahn's BFS topological sort; cycle exists iff order length < numCourses.
    Args:
        num_courses (int): total number of courses labeled 0 to n-1.
        prerequisites (List[List[int]]): each [a, b] means b must be taken before a.
    Returns:
        bool: True if all courses can be finished (no cycle), False otherwise.
    Time:  O(V+E) — V=num_courses, E=len(prerequisites)
    Space: O(V+E) — adj list + in-degree array + queue
    """
    adj = defaultdict(list)
    indegree = [0] * num_courses

    for course, prereq in prerequisites:
        adj[prereq].append(course)  # prereq → course (prereq must come first)
        indegree[course] += 1       # course gains one more dependency

    # start with all courses that have no prerequisites
    queue = deque(i for i in range(num_courses) if indegree[i] == 0)
    finished = 0

    while queue:
        course = queue.popleft()    # this course's prerequisites all done
        finished += 1
        for next_course in adj[course]:
            indegree[next_course] -= 1          # one dependency resolved
            if indegree[next_course] == 0:
                queue.append(next_course)        # all dependencies done — ready

    return finished == num_courses  # all courses processed → no cycle

# Slow motion on [[1,0],[2,0],[3,1],[3,2]], numCourses=4:
# Edges: 0→1, 0→2, 1→3, 2→3
# in-degrees: [0,1,1,2]  queue=[0]
# pop 0→finished=1, decrement in[1]=0,in[2]=0 → queue=[1,2]
# pop 1→finished=2, decrement in[3]=1         → queue=[2]
# pop 2→finished=3, decrement in[3]=0         → queue=[3]
# pop 3→finished=4                            → queue=[]
# finished=4 == 4 → True

def test_harness(fn):
    tests = [
        (2, [[1,0]], True),                   # simple: 0 → 1
        (2, [[1,0],[0,1]], False),             # cycle: 0→1→0
        (4, [[1,0],[2,0],[3,1],[3,2]], True),  # diamond DAG
        (1, [], True),                         # single course, no prereqs
        (3, [[0,1],[0,2],[1,2]], True),        # 2→1→0 no cycle
        (3, [[0,1],[1,2],[2,0]], False),       # 3-node cycle
    ]
    passed = 0
    for *inputs, expected in tests:
        n, prereqs = inputs
        got = fn(n, prereqs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | n={n} prereqs={prereqs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(can_finish)
print("can_finish defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Course Schedule II — Find Order — LC 210

---

```
PROBLEM:
  Same setup as LC 207, but return one valid course order instead of True/False.
  Return [] if impossible (cycle).

TRICK:
  Same Kahn's algorithm — just collect the order as you process the BFS queue.
  If cycle: return []. Otherwise return the collected order.

SLOW MOTION TRACE on numCourses=4, prerequisites=[[1,0],[2,0],[3,1],[3,2]]:

  (same as LC 207)
  queue starts: [0]
  order: [0, 1, 2, 3]  or  [0, 2, 1, 3]  (both valid, depends on queue order)
  len=4 == 4 → return [0, 1, 2, 3]

EXTENSION — DFS POST-ORDER VERSION:
  Same graph, but use DFS with WHITE/GRAY/BLACK coloring.
  Append node to result AFTER all its successors are done (post-order).
  Reverse the result at the end.
  A GRAY neighbor = back edge = cycle → return [].

KEY INSIGHT:
  LC 207 and LC 210 are identical problems — the only difference is
  whether you return bool or the actual order list.

TIME:  O(V+E)
SPACE: O(V+E)
```

In [ ]:
def find_order(num_courses, prerequisites):
    """
    LC 210 — Course Schedule II
    Approach: Kahn's BFS topological sort; collect order as queue is processed.
    Args:
        num_courses (int): total number of courses labeled 0 to n-1.
        prerequisites (List[List[int]]): each [a, b] means b must be taken before a.
    Returns:
        List[int]: one valid course order, or [] if a cycle exists.
    Time:  O(V+E) — each node and edge processed once in BFS
    Space: O(V+E) — adjacency list + in-degree array + queue + output
    """
    adj = defaultdict(list)
    indegree = [0] * num_courses

    for course, prereq in prerequisites:
        adj[prereq].append(course)  # prereq unlocks course
        indegree[course] += 1

    queue = deque(i for i in range(num_courses) if indegree[i] == 0)
    order = []                       # accumulate valid order here

    while queue:
        course = queue.popleft()     # prerequisites met — take this course
        order.append(course)         # record in the order we take courses
        for next_course in adj[course]:
            indegree[next_course] -= 1
            if indegree[next_course] == 0:
                queue.append(next_course)

    return order if len(order) == num_courses else []

# Also demonstrate the DFS post-order approach for completeness
def find_order_dfs(num_courses, prerequisites):
    """
    LC 210 — Course Schedule II (DFS post-order version)
    Approach: DFS with WHITE/GRAY/BLACK; post-order append then reverse.
    Time:  O(V+E)  Space: O(V+E)
    """
    adj = defaultdict(list)
    for course, prereq in prerequisites:
        adj[prereq].append(course)

    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * num_courses
    order = []
    cycle = [False]

    def dfs(node):
        if cycle[0]:
            return
        color[node] = GRAY            # entering this course — in-progress
        for nxt in adj[node]:
            if color[nxt] == GRAY:
                cycle[0] = True       # found a back edge — cycle exists
                return
            if color[nxt] == WHITE:
                dfs(nxt)
        color[node] = BLACK           # all successors done
        order.append(node)            # post-order: this course comes AFTER its successors

    for i in range(num_courses):
        if color[i] == WHITE:
            dfs(i)

    if cycle[0]:
        return []
    return order[::-1]               # reverse: post-order gives reverse topo order

# Slow motion Kahn's on [[1,0],[2,0],[3,1],[3,2]], n=4:
# queue=[0] → pop 0, order=[0], unlock 1(in=0) and 2(in=0) → queue=[1,2]
# pop 1, order=[0,1], unlock 3(in=1) → queue=[2]
# pop 2, order=[0,1,2], unlock 3(in=0) → queue=[3]
# pop 3, order=[0,1,2,3] → return [0,1,2,3]

def test_harness(fn):
    # note: multiple valid orders may exist — check the order IS a valid topo sort
    def is_valid_order(n, prereqs, order):
        if len(order) != n:
            return False
        pos = {course: i for i, course in enumerate(order)}
        for course, prereq in prereqs:
            if pos[prereq] >= pos[course]:  # prereq must appear before course
                return False
        return True

    tests = [
        (2, [[1,0]]),
        (2, [[1,0],[0,1]]),  # cycle → expect []
        (4, [[1,0],[2,0],[3,1],[3,2]]),
        (1, []),
        (3, [[0,1],[0,2],[1,2]]),
    ]
    passed = 0
    for n, prereqs in tests:
        got = fn(n, prereqs)
        has_cycle = any(True for a, b in prereqs if a == b) or \
                    (n == 2 and len(prereqs) == 2 and set(map(tuple,prereqs)) == {(0,1),(1,0)})
        # simpler: just check validity
        is_cycle_case = (got == [])
        if is_cycle_case:
            # verify it was indeed impossible
            bfs_check = can_finish(n, prereqs)
            ok = not bfs_check  # [] returned iff cycle
        else:
            ok = is_valid_order(n, prereqs, got)
        status = "PASSED" if ok else "FAILED"
        if status == "FAILED":
            print(f"{status} | n={n} prereqs={prereqs} | got={got}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")

print("Kahn's BFS:")
test_harness(find_order)
print("DFS post-order:")
test_harness(find_order_dfs)
print("find_order and find_order_dfs defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Alien Dictionary — LC 269

---

```
PROBLEM:
  Given a sorted list of words in an alien language, derive the character
  ordering of the alien alphabet. Return any valid ordering, or "" if impossible.

TRICK:
  Build a character dependency graph from adjacent word pairs.
  Compare word[i] and word[i+1] character by character:
    - First position where they differ: word[i][j] → word[i+1][j] (j comes before j+1)
    - If word[i] is a prefix of word[i+1] with word[i] longer: INVALID (prefix paradox)
  Then run topological sort on the character graph.

SLOW MOTION TRACE on words=["wrt","wrf","er","ett","rftt"]:

  Compare adjacent pairs:
  "wrt" vs "wrf": w==w, r==r, t≠f  →  t→f
  "wrf" vs "er":  w≠e              →  w→e
  "er"  vs "ett": e==e, r≠t        →  r→t
  "ett" vs "rftt": e≠r             →  e→r

  Graph edges: t→f, w→e, r→t, e→r
  Unique chars: {w, r, t, e, f}

  Kahn's topo sort on this character graph:
  in-degrees: w=0, r=1, t=1, e=1, f=1
  queue = [w]
  pop w → order="w", unlock e(in=0) → queue=[e]
  pop e → order="we", unlock r(in=0) → queue=[r]
  pop r → order="wer", unlock t(in=0) → queue=[t]
  pop t → order="wert", unlock f(in=0) → queue=[f]
  pop f → order="wertf"
  len=5 == 5 → return "wertf"

KEY INSIGHT:
  The problem IS topological sort — you just need to extract the edges
  from adjacent word comparisons first.

TIME:  O(C) where C = total characters across all words
SPACE: O(1) — at most 26 chars, constant-size graph
```

In [ ]:
def alien_order(words):
    """
    LC 269 — Alien Dictionary
    Approach: Extract char-ordering edges from adjacent word pairs, then Kahn's topo sort.
    Args:
        words (List[str]): list of words sorted by the alien language's alphabet.
    Returns:
        str: one valid character ordering, or '' if a cycle or invalid input exists.
    Time:  O(C) — C = total characters across all words (at most 26 unique chars)
    Space: O(1) — character graph has at most 26 nodes and 26^2 edges
    """
    # collect all unique characters in the alien alphabet
    adj = {ch: [] for word in words for ch in word}  # every char gets a slot
    indegree = {ch: 0 for word in words for ch in word}

    # extract ordering edges from each adjacent word pair
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        min_len = min(len(w1), len(w2))
        # check prefix paradox: "abc" before "ab" is INVALID in sorted order
        if len(w1) > len(w2) and w1[:min_len] == w2[:min_len]:
            return ""   # longer word listed first — contradicts sorted order
        for j in range(min_len):
            if w1[j] != w2[j]:          # first differing character gives us an edge
                adj[w1[j]].append(w2[j])  # w1[j] must come before w2[j]
                indegree[w2[j]] += 1
                break                    # only the FIRST difference matters per pair

    # Kahn's BFS topological sort on the character graph
    queue = deque(ch for ch in indegree if indegree[ch] == 0)
    order = []

    while queue:
        ch = queue.popleft()
        order.append(ch)
        for next_ch in adj[ch]:
            indegree[next_ch] -= 1
            if indegree[next_ch] == 0:
                queue.append(next_ch)

    if len(order) < len(indegree):
        return ""    # cycle detected — no valid alien alphabet exists
    return "".join(order)

# Slow motion on ["wrt","wrf","er","ett","rftt"]:
# edges: t→f, w→e, r→t, e→r
# in-degrees: {w:0, r:1, t:1, e:1, f:1}
# queue=[w] → pop w → order=w, e gets in=0 → queue=[e]
# pop e → order=we, r gets in=0 → queue=[r]
# pop r → order=wer, t gets in=0 → queue=[t]
# pop t → order=wert, f gets in=0 → queue=[f]
# pop f → order=wertf → return "wertf"

def test_harness(fn):
    def is_valid_alien_order(words, result):
        """Check that result is consistent with all adjacent word pair orderings."""
        if not result:
            return False
        pos = {ch: i for i, ch in enumerate(result)}
        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i+1]
            for j in range(min(len(w1), len(w2))):
                if w1[j] != w2[j]:
                    if pos.get(w1[j], -1) >= pos.get(w2[j], -1):
                        return False
                    break
        return True

    tests = [
        (["wrt","wrf","er","ett","rftt"], True),  # valid order exists
        (["z","x"], True),                         # z→x
        (["z","x","z"], False),                    # cycle z→x→z → ""
        (["abc","ab"], False),                     # prefix paradox → ""
        (["a","b","a"], False),                    # cycle: a→b→a
    ]
    passed = 0
    for words, expect_valid in tests:
        got = fn(words)
        if expect_valid:
            ok = is_valid_alien_order(words, got) if got else False
        else:
            ok = (got == "")
        status = "PASSED" if ok else "FAILED"
        if status == "FAILED":
            print(f"{status} | words={words} | got='{got}'")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")

test_harness(alien_order)
print("alien_order defined.")

<a id='8'></a>
## 8. The Topological Sort Decision Map

```
QUESTION TYPE                         KEY TECHNIQUE               LC PROBLEMS
─────────────────────────────────────────────────────────────────────────────
Can all tasks finish?                 Kahn's + len check           207
Find valid task order                 Kahn's BFS → collect order   210
Detect cycle (directed graph)         Kahn's (len<n) or DFS gray   207, 210
Character order from sorted words     Build char graph + topo      269
Build order / compilation order       Reverse edges + topo sort    —
Shortest path in DAG                  Topo order + edge relaxation  —
Count paths in DAG                    Topo order + DP forward      —
Minimum courses per semester          Layer-by-layer Kahn's BFS    630

KAHN'S vs DFS:
  Kahn's BFS:   cleaner, iterative, easy cycle detection (len check)
  DFS:          needed when you already have DFS structure (e.g., post-order)
  Both:         O(V+E) time, O(V+E) space

CYCLE DETECTION SIGNALS:
  Kahn's: len(order) < num_nodes → some nodes stuck at indegree > 0
  DFS:    encounter a GRAY (in-progress) node → back edge → cycle
```

<a id='9'></a>
## 9. Interview Cheat Sheet

**1. When to reach for Topological Sort:**

| Signal | What to Do |
|--------|------------|
| "prerequisites" or "dependencies" | Kahn's BFS topological sort |
| "valid order" or "build order" | Kahn's → collect and return order |
| "detect cycle in directed graph" | Kahn's: len(order)<n or DFS gray |
| "char ordering from sorted words" | Build char graph, run topo sort |
| "can all tasks be completed" | Kahn's: return len(order)==n |

**2. The O(V+E) operations — memorize these:**

```python
# BUILD GRAPH + IN-DEGREES from edge list
adj = defaultdict(list)
indegree = [0] * n
for a, b in edges:       # a → b (a must come before b)
    adj[a].append(b)
    indegree[b] += 1

# KAHN'S BFS CORE
queue = deque(i for i in range(n) if indegree[i] == 0)
order = []
while queue:
    node = queue.popleft()
    order.append(node)
    for nb in adj[node]:
        indegree[nb] -= 1
        if indegree[nb] == 0:
            queue.append(nb)
has_cycle = len(order) < n

# DFS POST-ORDER CORE
WHITE, GRAY, BLACK = 0, 1, 2
color = [WHITE] * n
def dfs(u):
    color[u] = GRAY
    for v in adj[u]:
        if color[v] == GRAY: return True   # cycle!
        if color[v] == WHITE:
            if dfs(v): return True
    color[u] = BLACK
    order.append(u)                        # post-order
    return False
# final: order[::-1] = topological order
```

**3. Common templates:**

```python
# TEMPLATE: CAN FINISH (LC 207)
def can_finish(n, prereqs):
    adj = defaultdict(list)
    indeg = [0]*n
    for a, b in prereqs:
        adj[b].append(a); indeg[a] += 1
    q = deque(i for i in range(n) if not indeg[i])
    done = 0
    while q:
        u = q.popleft(); done += 1
        for v in adj[u]:
            indeg[v] -= 1
            if not indeg[v]: q.append(v)
    return done == n

# TEMPLATE: ALIEN DICTIONARY — EXTRACT EDGES
for i in range(len(words)-1):
    w1, w2 = words[i], words[i+1]
    if len(w1) > len(w2) and w1[:len(w2)] == w2: return ""  # prefix check
    for c1, c2 in zip(w1, w2):
        if c1 != c2:
            adj[c1].append(c2); indeg[c2] += 1
            break
```

**4. Gotchas to not forget:**

```
❌  Forgetting to check len(order)==n for cycle detection in Kahn's
❌  Using topological sort on undirected graphs — it only applies to DAGs
❌  Building edges in the wrong direction (b→a when you want a→b)
❌  Missing the prefix paradox check in alien dictionary ("abc" before "ab" is invalid)
✅  Kahn's is usually cleaner and easier to implement than DFS in interviews
✅  Multiple valid topological orders exist — any one is acceptable
✅  in-degree = number of INCOMING edges = number of prerequisites that must be done first
✅  Nodes with in-degree 0 can start immediately (no dependencies)
```

## Summary Map

```
                    🗂️ TOPOLOGICAL SORT
                            │
             ┌──────────────┼──────────────┐
             │              │              │
         KAHN'S BFS      DFS POST-ORDER  BUILD GRAPH
             │              │              │
        ┌────┴────┐     ┌───┴───┐       EDGE LIST
        │         │     │       │       → adj list
     CAN FINISH  ORDER  DETECT  FIND    → indegree[]
     len(order)  collect CYCLE  ORDER
     == n?       as you  GRAY   reverse
     LC 207      pop     neighbor post-order
                 LC 210  LC 207   LC 210
                            │
                     ALIEN DICTIONARY
                     extract edges from
                     adjacent word pairs
                     → then Kahn's
                     LC 269

CORE RULE:
  Topological sort = valid ordering of a DAG where all dependencies come first.
  Cycle exists ↔ no valid topological order exists.
  Kahn's: in-degree queue. DFS: post-order + reverse.
```

---
*End of Topological Sort Master Guide — Sean Edition*